In [1]:
import time
_t0 = time.perf_counter()
def _step(label):
    print(f"[setup +{time.perf_counter()-_t0:.1f}s] {label}", flush=True)

_step("kernel is running; importing packages")
from pathlib import Path
import json
import re
import numpy as np
_step("NumPy imported")
import pandas as pd
_step("pandas imported")
import ollama
_step("ollama imported (model has not been loaded)")

BASE_DIR = Path(r"E:/Data_for_Practice/JapMedia")  # edit this path
ARTICLE_DIR = BASE_DIR / "data" / "embed_figure"
LABEL_DIRS = [BASE_DIR / "data" / "kor_data", BASE_DIR / "data" / "jap_data" / "translated"]
OUTPUT_DIR = BASE_DIR / "validation"
MODEL_NAME = "llama4"  # record exact locally installed model tag/digest
RANDOM_SEED = 42
TOP_K, RANDOM_K = 3, 2
MAX_CHARS_PER_ARTICLE = 3000
ARTICLE_COLUMNS = ["본문_en", "영문 번역", "본문", "article_text", "text"]
_step(f"checking article directory: {ARTICLE_DIR}")
article_files = sorted(ARTICLE_DIR.glob("*_doc_topics_with_emb.csv"))
_step(f"found {len(article_files)} article files")
if not article_files:
    raise FileNotFoundError(f"No article-level topic CSVs in {ARTICLE_DIR}")
_step(f"checking output directory: {OUTPUT_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_step("setup complete")

[setup +0.0s] kernel is running; importing packages
[setup +0.2s] NumPy imported
[setup +1.0s] pandas imported
[setup +1.8s] ollama imported (model has not been loaded)
[setup +1.8s] checking article directory: E:\Data_for_Practice\JapMedia\data\embed_figure
[setup +1.8s] found 12 article files
[setup +1.8s] checking output directory: E:\Data_for_Practice\JapMedia\validation
[setup +1.8s] setup complete


In [2]:
def make_items(path, rng):
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    body_col = next((c for c in ARTICLE_COLUMNS if c in df.columns), None)
    if not body_col:
        raise ValueError(f"{path.name}: original article body missing; available columns: {list(df.columns)[:30]}")
    for c in ["topic", "topic_label", "newspaper", "year"]:
        if c not in df.columns: raise ValueError(f"{path.name}: missing {c}")
    emb_cols = sorted([c for c in df.columns if re.fullmatch(r"emb_\d+", c)], key=lambda c: int(c[4:]))
    if not emb_cols: raise ValueError(f"{path.name}: no embedding columns")
    df["topic"] = pd.to_numeric(df["topic"], errors="coerce")
    df = df.loc[df["topic"].notna() & (df["topic"] != -1)].copy()
    items = []
    for topic, group in df.groupby("topic", sort=True):
        group = group.copy()
        x = group[emb_cols].to_numpy(dtype=float)
        centroid = x.mean(axis=0)
        den = np.linalg.norm(x, axis=1) * np.linalg.norm(centroid)
        group["centroid_similarity"] = np.divide(x @ centroid, den, out=np.full(len(group), np.nan), where=den > 0)
        group["original_row"] = group.index
        top_idx = group.sort_values(["centroid_similarity", "original_row"], ascending=[False, True], na_position="last").head(TOP_K).index.tolist()
        rest = group.drop(index=top_idx)
        random_idx = rng.choice(rest.index.to_numpy(), size=min(RANDOM_K,len(rest)), replace=False).tolist()
        articles = []
        for idx, selection in [(i,"centroid") for i in top_idx] + [(i,"random") for i in random_idx]:
            row = group.loc[idx]
            raw = "" if pd.isna(row[body_col]) else str(row[body_col])
            articles.append({"original_row":int(idx), "selection":selection,
                             "centroid_similarity":float(row.centroid_similarity),
                             "text":raw[:MAX_CHARS_PER_ARTICLE], "original_chars":len(raw),
                             "truncated":len(raw)>MAX_CHARS_PER_ARTICLE})
        items.append({"source_file":path.name, "newspaper":str(group.newspaper.iloc[0]),
                      "year":int(group.year.iloc[0]), "topic":int(topic),
                      "n_articles":len(group), "body_column":body_col,
                      "topic_keywords":str(group.topic_label.iloc[0]), "articles":articles})
    return items

rng = np.random.default_rng(RANDOM_SEED)
items = [item for path in article_files for item in make_items(path,rng)]
print(f"Retained topics: {len(items)}; article excerpts: {sum(len(x['articles']) for x in items)}")
print(f"Truncated article texts: {sum(a['truncated'] for x in items for a in x['articles'])}")
with open(OUTPUT_DIR / "frame_review_inputs.json", "w", encoding="utf-8") as f:
    json.dump(items,f,ensure_ascii=False,indent=2)

Retained topics: 43; article excerpts: 215
Truncated article texts: 156


In [5]:
FRAME_DEFINITIONS = """1. Economic Costs/Benefits (ECB): Economic trade-offs and distributional consequences of climate policy, including prices, employment, costs, competitiveness, and investment.
2. Technological Transition & Industrial Competition (TT-IC): Low-carbon technologies, their deployment and supply chains, industrial strategy, and technological competition.
3. Political Imbalance / Institutions & Geopolitics (PII-G): Institutional bargaining, governance, diplomacy, trade conflict, and geopolitics linked to climate or energy transition.
4. Climate Crisis Response & Justice (CCR-J): Climate impacts and risks, mitigation or adaptation as collective responses, vulnerability, equity, and just transition."""
ALLOWED = {"ECB","TT-IC","PII-G","CCR-J","INSUFFICIENT","MIXED"}

def make_prompt(item):
    snippets = "\n\n".join(
        f"Article {j} ({a['selection']}, original row {a['original_row']}; "
        f"{'truncated excerpt' if a['truncated'] else 'full saved text'}):\n{a['text']}"
        for j,a in enumerate(item['articles'],1))
    return f"""Assess a BERTopic topic from climate-related news using its representative terms AND the selected article texts. The unit of the final decision is the topic, not each article. The articles were selected from this topic and may vary. Do not infer problem definitions, causal claims, evaluations, or solutions unless supported by the article text. If no category is supported, choose INSUFFICIENT. If genuinely different frames coexist without a defensible dominant category, choose MIXED. Ignore previous LLM labels (none are supplied).

FRAME DEFINITIONS:
{FRAME_DEFINITIONS}

Newspaper: {item['newspaper']}; year: {item['year']}; topic: {item['topic']}; topic size: {item['n_articles']}
Representative terms: {item['topic_keywords']}

SELECTED ARTICLE TEXTS:
{snippets}

Return valid JSON with keys: frame (one of ECB, TT-IC, PII-G, CCR-J, INSUFFICIENT, MIXED), concise_label (2-7 words), rationale (brief), evidence (list of up to 3 objects with article_number, short_excerpt, and relevance), article_frames (one category per supplied article), and uncertainty (brief). Do not invent excerpts or facts."""

def run_one(item):
    prompt = make_prompt(item)
    result = ollama.chat(model=MODEL_NAME,
                         messages=[{"role":"system","content":"Return JSON only."},
                                   {"role":"user","content":prompt}],
                         format="json", options={"temperature":0,"num_predict":700,"seed":RANDOM_SEED})
    parsed = json.loads(result["message"]["content"])
    raw_frame = parsed.get("frame")
    if not isinstance(raw_frame, str):
        raise ValueError(f"Unexpected frame: {raw_frame}")
    # Normalize harmless formatting variation such as "PII- G".
    canonical = re.sub(r"\s+", "", raw_frame.upper().replace("–", "-").replace("—", "-"))
    if canonical not in ALLOWED:
        raise ValueError(f"Unexpected frame: {raw_frame}")
    parsed["frame_raw"] = raw_frame
    parsed["frame"] = canonical
    return parsed, prompt

# Restart-safe: preserve every successful call, including the exact prompt.
results_path = OUTPUT_DIR / "frame_reassessment_results.jsonl"
existing = {}
if results_path.exists():
    for line in results_path.read_text(encoding="utf-8").splitlines():
        rec = json.loads(line)
        existing[(rec["source_file"],rec["topic"])] = rec
for item in items:
    key = (item["source_file"],item["topic"])
    if key in existing: continue
    try:
        answer, prompt = run_one(item)
        rec = {"source_file":item["source_file"], "newspaper":item["newspaper"],
               "year":item["year"], "topic":item["topic"],
               "n_articles":item["n_articles"], "selected_article_rows":
               [a["original_row"] for a in item["articles"]], "model":MODEL_NAME,
               "prompt":prompt,"response":answer}
        with results_path.open("a",encoding="utf-8") as f:
            f.write(json.dumps(rec,ensure_ascii=False)+"\n")
        existing[key] = rec
        print(item["newspaper"],item["year"],item["topic"],answer["frame"])
    except Exception as exc:
        print("FAILED",key,repr(exc))
print(f"Completed {len(existing)} / {len(items)} topics")

아사히신문 2022 0 PII-G
아사히신문 2024 1 PII-G
Completed 43 / 43 topics


In [6]:
old_files = [p for d in LABEL_DIRS for p in d.glob("*bestframe_labels_specific*.csv")]
old_rows=[]
for path in old_files:
    data=pd.read_csv(path,encoding="utf-8-sig")
    needed={"file","Topic","best_frame","best_frame_label"}
    if not needed.issubset(data.columns): continue
    for _,r in data.iterrows():
        if pd.isna(r.Topic) or int(r.Topic)==-1: continue
        old_rows.append({"source_file":str(r['file']).replace("_topic_summary.csv","_doc_topics_with_emb.csv"),
                         "topic":int(r.Topic),"old_frame_raw":r.best_frame,
                         "old_label":r.best_frame_label})
old=pd.DataFrame(old_rows)

def normalize_frame(value):
    s=str(value).lower()
    if "economic" in s: return "ECB"
    if "technological" in s: return "TT-IC"
    if "political" in s or "geopolitic" in s: return "PII-G"
    if "climate crisis" in s: return "CCR-J"
    return "UNKNOWN"

records=[{"source_file":x['source_file'],"topic":x['topic'],"newspaper":x['newspaper'],
          "year":x['year'],"new_frame":x['response']['frame'],
          "new_label":x['response'].get('concise_label',''),
          "rationale":x['response'].get('rationale',''),
          "uncertainty":x['response'].get('uncertainty','')}
         for x in existing.values()]
comp=pd.DataFrame(records)
if not old.empty:
    comp=comp.merge(old,on=["source_file","topic"],how="left",validate="one_to_one")
    comp['old_frame']=comp.old_frame_raw.map(normalize_frame)
    comp['requires_review']=(comp.old_frame!=comp.new_frame) | comp.new_frame.isin(['MIXED','INSUFFICIENT'])
else:
    comp['requires_review']=True
comp.to_csv(OUTPUT_DIR / "frame_reassessment_comparison.csv",index=False,encoding="utf-8-sig")
print(f"Completed: {len(comp)}; flagged for human review: {comp.requires_review.sum()}")
display(comp.loc[comp.requires_review])

Completed: 43; flagged for human review: 10


,source_file,topic,newspaper,year,new_frame,new_label,rationale,uncertainty,old_frame_raw,old_label,old_frame,requires_review
5,(영문번역 추가)_요미우리신문_2022년도 데이터_doc_topics_with_em...,1,요미우리신문,2022,PII-G,Geopolitics and Institutional Relations,The topic is characterized by discussions of d...,Low,Climate Crisis Response & Justice Frame,Pacific Island Climate Resilience Declaration,CCR-J,True
7,(영문번역 추가)_요미우리신문_2024년도 데이터_doc_topics_with_em...,0,요미우리신문,2024,CCR-J,Climate Crisis Response & Justice,"The topic focuses on climate change impacts, m...",Some articles discuss technological transition...,Technological Transition,Japanese Nuclear Power Supply Chain,TT-IC,True
9,(영문번역 추가)_요미우리신문_2024년도 데이터_doc_topics_with_em...,2,요미우리신문,2024,PII-G,Climate Governance & Geopolitics,The topic focuses on international climate neg...,"The topic appears to have a dominant frame, bu...",Climate Crisis Response & Justice,Global Greenhouse Gas Emission Targets,CCR-J,True
16,20251112_중복기사 삭제완료_(영문번역 추가)_요미우리신문_2023년도 데이터...,2,요미우리신문,2023,CCR-J,Climate Crisis Response,Representative terms and article texts focus o...,Some articles discuss climate change in passin...,Political Imbalance / Institutions & Geopolitics,Party Election Dynamics,PII-G,True
18,조선일보_2022년도 데이터_translated_gpt_doc_topics_with...,0,조선일보,2022,PII-G,Geopolitics and Institutional Response,The topic primarily revolves around geopolitic...,"Moderate, due to the complexity of geopolitica...",Technological Transition,EV Battery Industry Competition,TT-IC,True
24,조선일보_2023년도 데이터_translated_gpt_doc_topics_with...,2,조선일보,2023,CCR-J,Climate Crisis Response & Justice,"The topic focuses on climate change impacts, r...",Some articles discuss technological and econom...,Technological Transition,Renewable Energy Industrial Transition,TT-IC,True
28,조선일보_2024년도 데이터_translated_gpt_doc_topics_with...,2,조선일보,2024,CCR-J,Climate Crisis Response & Justice,The topic focuses on international cooperation...,Low,Political Imbalance / Institutions & Geopolitics,Trilateral Nuclear Summit Diplomacy,PII-G,True
29,한겨레_2022년도 데이터_translated_gpt_doc_topics_with_...,0,한겨레,2022,CCR-J,Climate Crisis Response and Justice,"The topic focuses on climate crisis response, ...","Some articles discuss economic aspects, but cl...",Political Imbalance / Institutions & Geopolitics,Biden Administration Global Security,PII-G,True
37,한겨레_2024년도 데이터_translated_gpt_doc_topics_with_...,0,한겨레,2024,CCR-J,Climate Crisis Response and Justice,"The topic focuses on climate crisis response, ...","Some articles touch on economic aspects, but C...",Economic Costs/Benefits Frame,Climate Change Mitigation Finance,ECB,True
42,20251112_중복기사 삭제완료_(영문번역 추가)_아사히신문_2024년도 데이터_...,1,아사히신문,2024,PII-G,Politics and Geopolitics,The topic is dominated by discussions on inter...,low,Economic Costs/Benefits Frame,US Presidential Energy Policy Economics,ECB,True
